# Design-plan generation timings

One-off profiling notebook: point `INPUT_DIR` at a folder of source DXFs, run all cells top-to-bottom, and the timing tables drop into the bottom cell + into an Excel file.

Stages measured per drawing:

| Stage | What's inside |
|---|---|
| `extract` | DXF read + preprocessing + BERT classification |
| `coverings` | План покрытий (incl. exact-area computation via C++ helper) |
| `landscaping` | План озеленения |
| `maf` | План МАФ (incl. meta-block decomposition) |
| `layout` | Разбивочный план (incl. dimensions, anti-overlap labels) |
| `total` | Sum of all stages + per-drawing overhead |

In [2]:
# Imports + bootstrap the local `pipeline` package.
import sys
import time
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from pipeline import (
    _extract, plan_coverings, plan_landscaping, plan_maf, plan_layout,
    DEFAULT_MODEL_DIR,
)

STAGES = ['extract', 'coverings', 'landscaping', 'maf', 'layout']
print(f'pipeline imported. model dir = {DEFAULT_MODEL_DIR}')

pipeline imported. model dir = C:\Users\artem\MAIN\projects\plain\local\python_modules\ai_models


In [ ]:
INPUT_DIR = Path(r'C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_design')
EXCEL_PATH = HERE / 'timings' / 'generation_timings.xlsx'
EXCEL_PATH.parent.mkdir(exist_ok=True, parents=True)

# True  → burn BERT cold-start on the first drawing without recording it.
# False → record every drawing as-is. The first drawing's `extract` time
#         will include the ~30 s BERT load.
DO_WARMUP = True

print(f'Input dir : {INPUT_DIR}')
print(f'Excel out : {EXCEL_PATH}')
print(f'Warmup    : {DO_WARMUP}')

Input dir : C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_design
Excel out : c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\timings\generation_timings.xlsx
Warmup    : True


In [ ]:
# Find input DXFs
dxf_paths = sorted(p for p in INPUT_DIR.glob('*.dxf') if p.is_file())
if not dxf_paths:
    raise FileNotFoundError(f'No .dxf files found directly in {INPUT_DIR}')

print(f'Found {len(dxf_paths)} DXF file(s):')
for p in dxf_paths:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'  {p.name:<40s}  {size_mb:6.2f} MB')

Found 10 DXF file(s):
  _12.5_Сквер Моргунова_Генплан.dxf          17.08 MB
  _13.1_генплан 1ч.dxf                       54.58 MB
  _17.1_4_Генеральный план.dxf                8.00 MB
  _22.3_2 ген.план.dxf                       19.13 MB
  _31_Комплект_СПОЗУ3.dxf                     1.24 MB
  _33.3_15_спозу_генплан.dxf                 45.29 MB
  _33.4_15_спозу_генплан.dxf                 46.01 MB
  _33.5_15.1-15.3_спозу_генплан.dxf          46.75 MB
  _33.8_15_спозу_генплан.dxf                 45.09 MB
  design_1.dxf                               13.48 MB


In [ ]:
# Per-drawing timing function. Runs the full pipeline and returns
def time_one_drawing(input_path, model_dir=DEFAULT_MODEL_DIR, verbose=True):
    input_path = Path(input_path).resolve()
    output_dir = input_path.parent / input_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = input_path.stem
    if verbose:
        print(f'\n=== {input_path.name} ===')

    times = {'drawing': input_path.name}
    total_t0 = time.perf_counter()

    # 1) extract + classify
    t0 = time.perf_counter()
    dxf_path, classification, _ = _extract.extract_and_classify(
        str(input_path),
        model_dir=model_dir,
        csv_path=str(output_dir / f'{stem}.design.classes.csv'),
    )
    times['extract'] = time.perf_counter() - t0
    if verbose: print(f'  extract+classify  {times["extract"]:7.2f}s')

    src_doc = _extract.load_source_doc(dxf_path)

    # 2-5) each plan generator
    plans = [
        ('coverings',   plan_coverings,   f'{stem}_coverings.dxf'),
        ('landscaping', plan_landscaping, f'{stem}_landscaping.dxf'),
        ('maf',         plan_maf,         f'{stem}_maf.dxf'),
        ('layout',      plan_layout,      f'{stem}_layout.dxf'),
    ]
    for label, mod, fname in plans:
        out = str(output_dir / fname)
        t0 = time.perf_counter()
        mod.generate(src_doc, classification, out)
        times[label] = time.perf_counter() - t0
        if verbose: print(f'  {label:<16s}  {times[label]:7.2f}s')

    times['total'] = time.perf_counter() - total_t0
    if verbose: print(f'  {"TOTAL":<16s}  {times["total"]:7.2f}s')
    return times

print('time_one_drawing() defined.')

time_one_drawing() defined.


In [ ]:
if DO_WARMUP:
    print('Warmup pass (timings NOT recorded)...')
    _ = time_one_drawing(dxf_paths[0], verbose=True)
    print('\n--- warmup done; recording starts on the next cell ---')
else:
    print('Warmup skipped — first drawing will include BERT cold-start time.')

Warmup pass (timings NOT recorded)...

=== _12.5_Сквер Моргунова_Генплан.dxf ===
[DES extract] Input: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_design\_12.5_Сквер Моргунова_Генплан.dxf
[DES extract] DXF: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_design\_12.5_Сквер Моргунова_Генплан.dxf
[DES extract] Extracted objects: 2424
[preproc] dynamic-block map: 8 *U... → real-name entries
[preproc] topo-codes loaded: 192 entries from C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\base_plan\2.0\topo_codes_classification.xlsx
[preproc] auto-named blocks normalised: 7
[preproc] dynamic blocks resolved:      0
[preproc] CAD-formatted texts cleaned:  0
[preproc] topo-code preassignments:     0
[DES extract] Preprocessing done (0 rows preassigned via topo codes)
Device: cpu
  design: bert=intfloat/multilingual-e5-small, l1=12, l2=93 | mask 103/1116 pairs
  loading tokenizer: intfloat/multilingual-e

In [ ]:
# Main loop — recorded timings for every drawing in INPUT_DIR.
raw_rows = []
for i, p in enumerate(dxf_paths, start=1):
    print(f'\n[{i}/{len(dxf_paths)}] timing {p.name}')
    try:
        raw_rows.append(time_one_drawing(p, verbose=True))
    except Exception as e:
        print(f'  FAILED on {p.name}: {type(e).__name__}: {e}')

        failed = {'drawing': p.name}
        for s in STAGES + ['total']:
            failed[s] = float('nan')
        raw_rows.append(failed)

print(f'\n{len(raw_rows)} drawing(s) processed.')


[1/10] timing _12.5_Сквер Моргунова_Генплан.dxf

=== _12.5_Сквер Моргунова_Генплан.dxf ===
[DES extract] Input: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_design\_12.5_Сквер Моргунова_Генплан.dxf
[DES extract] DXF: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_design\_12.5_Сквер Моргунова_Генплан.dxf
[DES extract] Extracted objects: 2424
[preproc] dynamic-block map: 8 *U... → real-name entries
[preproc] auto-named blocks normalised: 7
[preproc] dynamic blocks resolved:      0
[preproc] CAD-formatted texts cleaned:  0
[preproc] topo-code preassignments:     0
[DES extract] Preprocessing done (0 rows preassigned via topo codes)
Device: cpu
  design: bert=intfloat/multilingual-e5-small, l1=12, l2=93 | mask 103/1116 pairs
  loading tokenizer: intfloat/multilingual-e5-small
[DES extract] Classified: 2424
[DES extract] Class table: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\ti

In [ ]:
wide_cols = ['drawing'] + STAGES + ['total']
df_wide = pd.DataFrame(raw_rows)[wide_cols].round(2)
df_wide

,drawing,extract,coverings,landscaping,maf,layout,total
0,_12.5_Сквер Моргунова_Генплан.dxf,30.21,25.76,1.12,0.35,1.61,70.37
1,_13.1_генплан 1ч.dxf,29.84,41.62,0.26,0.23,3.56,94.59
2,_17.1_4_Генеральный план.dxf,27.92,8.86,0.19,0.72,35.69,76.72
3,_22.3_2 ген.план.dxf,25.15,28.47,0.33,0.16,0.10,65.07
4,_31_Комплект_СПОЗУ3.dxf,5.37,1.50,0.07,0.06,0.15,7.55
5,_33.3_15_спозу_генплан.dxf,23.09,33.33,0.46,0.25,0.44,71.06
6,_33.4_15_спозу_генплан.dxf,27.82,33.44,0.41,6.42,1.16,82.73
7,_33.5_15.1-15.3_спозу_генплан.dxf,32.87,34.37,0.81,0.40,2.88,85.18
8,_33.8_15_спозу_генплан.dxf,22.84,31.78,0.37,0.17,0.86,69.02
9,design_1.dxf,16.91,12.35,0.24,0.30,1.00,35.82


In [ ]:
long_rows = []
for r in raw_rows:
    for stage in STAGES + ['total']:
        long_rows.append({
            'drawing': r['drawing'],
            'stage':   stage,
            'seconds': round(r.get(stage, float('nan')), 2),
        })
df_long = pd.DataFrame(long_rows)
df_long

,drawing,stage,seconds
0,_12.5_Сквер Моргунова_Генплан.dxf,extract,30.21
1,_12.5_Сквер Моргунова_Генплан.dxf,coverings,25.76
2,_12.5_Сквер Моргунова_Генплан.dxf,landscaping,1.12
3,_12.5_Сквер Моргунова_Генплан.dxf,maf,0.35
4,_12.5_Сквер Моргунова_Генплан.dxf,layout,1.61
5,_12.5_Сквер Моргунова_Генплан.dxf,total,70.37
6,_13.1_генплан 1ч.dxf,extract,29.84
7,_13.1_генплан 1ч.dxf,coverings,41.62
8,_13.1_генплан 1ч.dxf,landscaping,0.26
9,_13.1_генплан 1ч.dxf,maf,0.23


In [ ]:
# Per-stage summary across all drawings — mean / median / min / max.
if len(raw_rows) > 1:
    df_summary = (df_long.groupby('stage')['seconds']
                          .agg(['mean', 'median', 'min', 'max', 'std'])
                          .round(2)
                          .reindex(STAGES + ['total']))
    display(df_summary)
else:
    df_summary = None
    print('Only one drawing — summary stats are just the values themselves; skipping.')

,mean,median,min,max,std
stage,,,,,
extract,24.20,26.48,5.37,32.87,8.04
coverings,25.15,30.12,1.50,41.62,13.06
landscaping,0.43,0.35,0.07,1.12,0.31
maf,0.91,0.28,0.06,6.42,1.95
layout,4.74,1.08,0.10,35.69,10.93
total,65.81,70.72,7.55,94.59,25.74


In [ ]:
# Save to Excel
with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df_wide.to_excel(writer, sheet_name='wide',  index=False)
    df_long.to_excel(writer, sheet_name='long',  index=False)
    if df_summary is not None:
        df_summary.to_excel(writer, sheet_name='summary')

print(f'Saved: {EXCEL_PATH}')
print(f'  sheet "wide"    — {len(df_wide)} rows')
print(f'  sheet "long"    — {len(df_long)} rows')
if df_summary is not None:
    print(f'  sheet "summary" — {len(df_summary)} rows')

Saved: c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\timings\generation_timings.xlsx
  sheet "wide"    — 10 rows
  sheet "long"    — 60 rows
  sheet "summary" — 6 rows


## Area statistics

For every input drawing, compute its **concave-hull area** via
`calculate_area.calculate_dxf_area`. One value per drawing — useful as
a "size of the site" descriptor next to the timings (so a slow drawing
can be related to its physical area).

Generated-plan areas are NOT included here — they were noise. Time of
the area pass itself is also **not** recorded; only area values land in
the table. Result is written to a SEPARATE Excel file
(`areas/area_stats.xlsx`), independent of the timings file above.

In [ ]:
# Area statistics
from calculate_area import calculate_dxf_area

AREAS_EXCEL_PATH = HERE / 'areas' / 'area_stats.xlsx'
AREAS_EXCEL_PATH.parent.mkdir(exist_ok=True, parents=True)


def _safe_area(path):
    """Run calculate_dxf_area; coerce errors / missing files to NaN."""
    if not path.exists():
        return float('nan')
    _, value = calculate_dxf_area(str(path))
    if isinstance(value, (int, float)):
        return float(value)
    print(f'  ! {path.name}: {value}')
    return float('nan')


area_rows = []
for p in dxf_paths:
    print(f'Area for {p.name}...')
    area_rows.append({
        'drawing':     p.name,
        'source_area': _safe_area(p),
    })

df_areas = pd.DataFrame(area_rows)
df_areas['source_area'] = df_areas['source_area'].round(2)

# Across-drawings summary stats (only meaningful with >1 file).
if len(area_rows) > 1:
    df_areas_summary = (df_areas['source_area']
                          .agg(['mean', 'median', 'min', 'max', 'std'])
                          .round(2)
                          .rename_axis('stat')
                          .reset_index(name='source_area_m2'))
else:
    df_areas_summary = None

print('\n── Source drawing areas, m² (concave hull) ──')
display(df_areas)

if df_areas_summary is not None:
    print('\n── Across-drawings summary ──')
    display(df_areas_summary)

# Save to a SEPARATE Excel file
with pd.ExcelWriter(AREAS_EXCEL_PATH, engine='openpyxl') as writer:
    df_areas.to_excel(writer, sheet_name='areas_m2', index=False)
    if df_areas_summary is not None:
        df_areas_summary.to_excel(writer, sheet_name='summary', index=False)

print(f'\nSaved: {AREAS_EXCEL_PATH}')
print(f'  sheet "areas_m2" — {len(df_areas)} rows')
if df_areas_summary is not None:
    print(f'  sheet "summary"  — {len(df_areas_summary)} rows')

Area for _12.5_Сквер Моргунова_Генплан.dxf...
Area for _13.1_генплан 1ч.dxf...
Area for _17.1_4_Генеральный план.dxf...
Area for _22.3_2 ген.план.dxf...
Area for _31_Комплект_СПОЗУ3.dxf...
Area for _33.3_15_спозу_генплан.dxf...
Area for _33.4_15_спозу_генплан.dxf...
Area for _33.5_15.1-15.3_спозу_генплан.dxf...
Area for _33.8_15_спозу_генплан.dxf...
Area for design_1.dxf...

── Source drawing areas, m² (concave hull) ──


,drawing,source_area
0,_12.5_Сквер Моргунова_Генплан.dxf,6941.83
1,_13.1_генплан 1ч.dxf,128324.58
2,_17.1_4_Генеральный план.dxf,170652.53
3,_22.3_2 ген.план.dxf,513.31
4,_31_Комплект_СПОЗУ3.dxf,246.36
5,_33.3_15_спозу_генплан.dxf,14995.81
6,_33.4_15_спозу_генплан.dxf,25942.66
7,_33.5_15.1-15.3_спозу_генплан.dxf,56693.58
8,_33.8_15_спозу_генплан.dxf,9060.35
9,design_1.dxf,24802.46



── Across-drawings summary ──


,stat,source_area_m2
0,mean,43817.35
1,median,19899.13
2,min,246.36
3,max,170652.53
4,std,58923.40



Saved: c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\areas\area_stats.xlsx
  sheet "areas_m2" — 10 rows
  sheet "summary"  — 5 rows
